In [2]:
import sqlite3
import pandas as pd
import numpy as np
from pathlib import Path

# Anchor to project root and connect to the database
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'Notebook' else Path.cwd()
DB_PATH = PROJECT_ROOT / 'data' / 'clinical_trials.db'

conn = sqlite3.connect(DB_PATH)

print("Connected. Studies:", pd.read_sql("SELECT COUNT(*) AS n FROM studies", conn)['n'][0])


Connected. Studies: 500


# SQL Analysis — Business Questions

Analysis of the clinical trials dataset addressing the five core business questions.
Data quality findings from `01_data_profiling.ipynb` inform the scoping and caveats
throughout.

**Key quality constraints carried into this analysis:**
- Phase analysis is scoped to 201 interventional studies with a real phase (40.2% of cohort)
- `status = UNKNOWN` for 84 studies (16.8%) — completion outcome unverifiable
- Geographic analysis limited to 455 studies with location data (91%)
- 209 studies have partial dates (YYYY-MM) — duration calculations imprecise

---

## Q3 — Completion & Data Reliability

**Question:** Which factors are associated with higher trial completion rates?
Are any of these patterns confounded by data quality issues?

In [9]:


query = """
WITH status_counts AS (
    SELECT
        phase,
        COUNT(*)                                                  AS total_trials,
        SUM(CASE WHEN status = 'COMPLETED' THEN 1 ELSE 0 END)     AS completed,
        SUM(CASE WHEN status = 'UNKNOWN' THEN 1 ELSE 0 END)       AS unknown_status
    FROM studies
    WHERE study_type = 'INTERVENTIONAL'
      AND phase IS NOT NULL
      AND phase != 'NA'
    GROUP BY phase
)
SELECT
    phase,
    total_trials,
    completed,
    unknown_status,
    ROUND(100.0 * completed / total_trials, 1)                        AS naive_completion_rate,
    ROUND(100.0 * completed / NULLIF(total_trials - unknown_status, 0), 1) AS adjusted_completion_rate,
    ROUND(100.0 * completed / NULLIF(total_trials - unknown_status, 0), 1)
      - ROUND(100.0 * completed / total_trials, 1)                    AS rate_difference,
    ROUND(100.0 * unknown_status / total_trials, 1)                   AS pct_unknown
FROM status_counts
ORDER BY phase
"""
pd.read_sql(query, conn)

,phase,total_trials,completed,unknown_status,naive_completion_rate,adjusted_completion_rate,rate_difference,pct_unknown
0,EARLY_PHASE1,3,1,1,33.3,50.0,16.7,33.3
1,PHASE1,66,41,9,62.1,71.9,9.8,13.6
2,PHASE2,52,26,8,50.0,59.1,9.1,15.4
3,PHASE3,37,21,3,56.8,61.8,5.0,8.1
4,PHASE4,43,26,10,60.5,78.8,18.3,23.3


### Q3 Findings — Completion rates are confounded by unverifiable status

**Scope:** 201 interventional studies with a determinate phase (40.2% of cohort). Phase
analysis excludes observational studies (no phase by design) and interventional trials
marked "NA" (non-pharmaceutical interventions outside the Phase 1–4 framework).

**The confound:** 84 studies (16.8%) carry `status = UNKNOWN` — the registry has lost
track of their outcome. Completion rate therefore depends on a methodological choice:

- **Naive rate:** treats UNKNOWN as "not completed" → penalises phases with poor record upkeep
- **Adjusted rate:** excludes UNKNOWN from the denominator → assumes unknowns resemble knowns

| Phase | Naive | Adjusted | Swing | % Unknown |
|-------|-------|----------|-------|-----------|
| PHASE4 | 60.5% | 78.8% | **+18.3** | 23.3% |
| PHASE1 | 62.1% | 71.9% | +9.8 | 13.6% |
| PHASE3 | 56.8% | 61.8% | +5.0 | 8.1% |
| PHASE2 | 50.0% | 59.1% | +9.1 | 15.4% |
| EARLY_PHASE1 | 33.3% | 50.0% | +16.7 | 33.3% |

**The conclusion reverses:** naive calculation ranks PHASE1 highest (62.1%); adjusted
calculation ranks PHASE4 highest (78.8%). The data quality issue does not merely add noise —
it determines the answer.

**Why the effect is uneven:** the swing tracks each phase's proportion of unknown-status
trials. This makes the bias *systematic*, not random — phases with weaker registry
maintenance appear to perform worse. Hypothesis: Phase 4 (post-marketing surveillance) is
long-running and lower-priority for sponsor updates than pivotal Phase 2/3 trials that gate
approval, so its records go stale more often. This warrants verification against
`lastUpdatePostDate` rather than being asserted.

**Caveat:** EARLY_PHASE1 (n=3) is shown for completeness but excluded from interpretation —
the sample cannot support inference.

**Recommendation:** report completion rates as a range with explicit disclosure of the
unknown-status proportion, rather than a single point estimate. Neither calculation is
authoritative; presenting one without the other is misleading.

In [14]:
query = """
SELECT
    status,
    SUM(CASE WHEN phase = 'EARLY_PHASE1' THEN 1 ELSE 0 END) AS early_phase1,
    SUM(CASE WHEN phase = 'PHASE1'       THEN 1 ELSE 0 END) AS phase1,
    SUM(CASE WHEN phase = 'PHASE2'       THEN 1 ELSE 0 END) AS phase2,
    SUM(CASE WHEN phase = 'PHASE3'       THEN 1 ELSE 0 END) AS phase3,
    SUM(CASE WHEN phase = 'PHASE4'       THEN 1 ELSE 0 END) AS phase4,
    COUNT(*)                                                AS total
FROM studies
WHERE study_type = 'INTERVENTIONAL'
  AND phase IS NOT NULL
  AND phase != 'NA'
GROUP BY status
ORDER BY total DESC
"""
pd.read_sql(query, conn)

,status,early_phase1,phase1,phase2,phase3,phase4,total
0,COMPLETED,1,41,26,21,26,115
1,UNKNOWN,1,9,8,3,10,31
2,TERMINATED,0,4,7,7,1,19
3,RECRUITING,0,5,5,2,3,15
4,WITHDRAWN,0,5,2,1,2,10
5,ACTIVE_NOT_RECRUITING,0,1,3,2,0,6
6,NOT_YET_RECRUITING,1,1,1,1,1,5


In [13]:
# Get the long-format data with SQL
df = pd.read_sql("""
    SELECT status, phase, COUNT(*) AS n
    FROM studies
    WHERE study_type = 'INTERVENTIONAL' AND phase IS NOT NULL AND phase != 'NA'
    GROUP BY status, phase
""", conn)

# Pivot with pandas
pivot = df.pivot(index='status', columns='phase', values='n').fillna(0).astype(int)

# Add totals
pivot['TOTAL'] = pivot.sum(axis=1)
pivot.loc['TOTAL'] = pivot.sum()

pivot

phase,EARLY_PHASE1,PHASE1,PHASE2,PHASE3,PHASE4,TOTAL
status,,,,,,
ACTIVE_NOT_RECRUITING,0,1,3,2,0,6
COMPLETED,1,41,26,21,26,115
NOT_YET_RECRUITING,1,1,1,1,1,5
RECRUITING,0,5,5,2,3,15
TERMINATED,0,4,7,7,1,19
UNKNOWN,1,9,8,3,10,31
WITHDRAWN,0,5,2,1,2,10
TOTAL,3,66,52,37,43,201


### Status × Phase cross-tabulation

Scope: 201 interventional trials with a determinate phase.

**Termination risk rises through the pipeline, peaking at Phase 3:**
| Phase | Terminated | % of phase |
|-------|-----------|------------|
| PHASE1 | 4 | 6.1% |
| PHASE2 | 7 | 13.5% |
| PHASE3 | 7 | **18.9%** |
| PHASE4 | 1 | 2.3% |

Consistent with development economics: Phase 3 is the most capital-intensive stage, so
sponsors terminate on interim futility rather than absorb further cost. Phase 1 is cheap
enough to complete; Phase 4 drugs are already approved and rarely stopped.

**Two distinct failure modes at opposite ends of the pipeline:**
- **WITHDRAWN** (cancelled pre-enrolment) is front-loaded: 7.6% in Phase 1 vs 2.7% in Phase 3
- **TERMINATED** (stopped mid-flight) is back-loaded: 6.1% in Phase 1 vs 18.9% in Phase 3

Speculative early trials are abandoned before they start; late-stage trials fail after
significant investment.

**Record quality tracks regulatory consequence — supporting the Q3 confound hypothesis:**
| Phase | Unknown status | % of phase |
|-------|---------------|------------|
| PHASE3 | 3 | 8.1% |
| PHASE4 | 10 | **23.3%** |

Phase 3 data gates FDA approval and is maintained meticulously. Phase 4 (post-marketing)
has no equivalent gatekeeper, and its registry entries go stale nearly 3× as often. This
strengthens the Q3 finding that apparent phase performance differences partly reflect
**registry maintenance quality rather than trial execution**.

**Caveat:** cells with n < 10 (EARLY_PHASE1; NOT_YET_RECRUITING; ACTIVE_NOT_RECRUITING)
are shown for completeness but excluded from interpretation.

In [4]:
query = """
SELECT
    c.condition_name,
    COUNT(DISTINCT c.study_id) AS trial_count
FROM conditions c
GROUP BY c.condition_name
ORDER BY trial_count DESC
LIMIT 25
"""
pd.read_sql(query, conn)

,condition_name,trial_count
0,Healthy,12
1,Breast Cancer,8
2,COVID-19,6
3,Hypertension,6
4,Cardiovascular Diseases,5
5,Lung Cancer,5
6,Stroke,5
7,Depression,4
8,Healthy Volunteers,4
9,Heart Diseases,4


In [25]:
query = """
SELECT
        SUBSTR(start_date, 1, 4)            AS start_year,
        phase,
        COUNT(*)                            AS trial_count
    FROM studies
    WHERE start_date IS NOT NULL
      AND study_type = 'INTERVENTIONAL'
      AND phase IS NOT NULL
      AND phase != 'NA'
      --AND SUBSTR(start_date, 1, 4) >= '2010'
    GROUP BY start_year, phase
    order by start_year desc
"""
pd.read_sql(query, conn)

,start_year,phase,trial_count
0,2026,EARLY_PHASE1,1
1,2026,PHASE1,1
2,2026,PHASE3,1
3,2026,PHASE4,3
4,2025,PHASE1,2
...,...,...,...
85,1998,PHASE3,2
86,1996,PHASE2,1
87,1995,PHASE2,1
88,1983,PHASE2,1


In [20]:
query = """
WITH yearly AS (
    SELECT
        SUBSTR(start_date, 1, 4)            AS start_year,
        phase,
        COUNT(*)                            AS trial_count
    FROM studies
    WHERE start_date IS NOT NULL
      AND study_type = 'INTERVENTIONAL'
      AND phase IS NOT NULL
      AND phase != 'NA'
      --AND SUBSTR(start_date, 1, 4) >= '2010'
    GROUP BY start_year, phase
)
SELECT
    start_year,
    phase,
    trial_count,
    SUM(trial_count) OVER (PARTITION BY start_year)                       AS year_total,
    ROUND(100.0 * trial_count / SUM(trial_count) OVER (PARTITION BY start_year), 1) AS pct_of_year,
    SUM(trial_count) OVER (PARTITION BY phase ORDER BY start_year)        AS cumulative_for_phase
FROM yearly
ORDER BY start_year, phase
"""
pd.read_sql(query, conn)

,start_year,phase,trial_count,year_total,pct_of_year,cumulative_for_phase
0,1973,PHASE3,1,1,100.0,1
1,1983,PHASE2,1,1,100.0,1
2,1995,PHASE2,1,1,100.0,2
3,1996,PHASE2,1,1,100.0,3
4,1998,PHASE3,2,2,100.0,3
...,...,...,...,...,...,...
85,2025,PHASE4,2,8,25.0,40
86,2026,EARLY_PHASE1,1,6,16.7,3
87,2026,PHASE1,1,6,16.7,66
88,2026,PHASE3,1,6,16.7,37


In [23]:
query = """
SELECT
    SUBSTR(start_date, 1, 4) AS start_year,
    COUNT(*)                 AS trials_started,
    SUM(CASE WHEN study_type = 'INTERVENTIONAL' THEN 1 ELSE 0 END) AS interventional,
    SUM(CASE WHEN study_type = 'OBSERVATIONAL'  THEN 1 ELSE 0 END) AS observational
FROM studies
WHERE start_date IS NOT NULL
  AND SUBSTR(start_date, 1, 4) >= '2005'
GROUP BY start_year
ORDER BY start_year
"""
pd.read_sql(query, conn)

,start_year,trials_started,interventional,observational
0,2005,11,11,0
1,2006,5,4,1
2,2007,7,6,1
3,2008,13,10,3
4,2009,10,8,2
5,2010,12,10,2
6,2011,15,12,3
7,2012,15,13,2
8,2013,16,12,4
9,2014,24,22,2


### Q2 — Trial Landscape Overview

**Therapeutic areas — free-text fragmentation defect:**

The top conditions reveal a data-quality issue absent from the API's controlled fields:

| Labels | Trials | Issue |
|--------|--------|-------|
| COVID-19 / Covid19 / Coronavirus | 6 + 3 + 3 = 12 | One disease, three labels |
| Healthy / Healthy Volunteers | 12 + 4 = 16 | One concept, two labels |

`status` and `phase` (API-controlled vocabularies) showed zero inconsistency. `condition_name`
is free-text sponsor input and fragments accordingly — a naive top-conditions ranking
undercounts COVID trials by ~50%. The schema anticipates this via a `mesh_term` column
(MeSH = the NLM's controlled medical vocabulary), but API v2 does not expose it, leaving the
normalization layer unfilled (data gap #4). **Therapeutic-area analysis therefore requires
manual condition mapping and should be treated as approximate.**

**Landscape composition:** oncology dominates (Breast Cancer 8, Lung Cancer 5, Prostate 4,
Solid Tumors 4, Cancer 3, Lymphoma 3), followed by cardiovascular (Hypertension 6, CVD 5,
Stroke 5, Heart Disease 4, Heart Failure 4) — consistent with global research funding
priorities.

**Temporal pattern — observational share spiked during COVID:**

| Year | Total | Observational | Obs % |
|------|-------|---------------|-------|
| 2019 | 30 | 6 | 20% |
| 2020 | 34 | 11 | **32%** |
| 2021 | 37 | 11 | 30% |
| 2023 | 29 | 5 | 17% |

Observational studies' share rose ~60% in 2020-21 then reverted — plausibly reflecting
pandemic constraints on interventional trial conduct (site access, patient recruitment)
alongside urgent demand for real-world evidence.

**Critical caveat on trend interpretation:** trial starts appear to double from ~2013 to
~2019 within this sample. This **cannot be attributed to real-world growth** — the API
returned 500 studies without documented randomization or stratification, so sample recency
bias would produce an identical pattern. Registry-wide trend claims require a full extract
or a documented sampling design. Years 2026-27 are excluded from interpretation
(forward-dated, not-yet-recruiting records).

**Small-n caveat:** pre-2005 years contain 1-2 trials each; per-year percentages there are
not interpretable.

In [3]:
query = """
Select
  sum(enrollment) as enrollement,
  study_type,
  status
from studies
    group by study_type, status
order by enrollement desc
    limit 30
"""
pd.read_sql(query,conn)

,enrollement,study_type,status
0,817749.0,OBSERVATIONAL,UNKNOWN
1,86131.0,INTERVENTIONAL,COMPLETED
2,61824.0,INTERVENTIONAL,RECRUITING
3,36509.0,OBSERVATIONAL,COMPLETED
4,21676.0,OBSERVATIONAL,RECRUITING
5,9680.0,INTERVENTIONAL,UNKNOWN
6,2876.0,OBSERVATIONAL,NOT_YET_RECRUITING
7,2770.0,INTERVENTIONAL,ACTIVE_NOT_RECRUITING
8,2594.0,INTERVENTIONAL,TERMINATED
9,1579.0,INTERVENTIONAL,ENROLLING_BY_INVITATION


In [4]:
query = """
SELECT
    study_type,
    COUNT(*)                    AS n,
    MIN(enrollment)             AS min_val,
    MAX(enrollment)             AS max_val,
    ROUND(AVG(enrollment), 1)   AS mean_val
FROM studies
WHERE enrollment IS NOT NULL AND study_type IS NOT NULL
GROUP BY study_type
ORDER BY n DESC
"""
pd.read_sql(query, conn)

,study_type,n,min_val,max_val,mean_val
0,INTERVENTIONAL,396,0,50000,418.9
1,OBSERVATIONAL,97,0,800000,9072.3


In [5]:
df = pd.read_sql("""
    SELECT study_id, nct_id, status, study_type, phase, enrollment, title
    FROM studies
    WHERE enrollment IS NOT NULL
""", conn)

Q1 = df['enrollment'].quantile(0.25)
Q3 = df['enrollment'].quantile(0.75)
IQR = Q3 - Q1
lower_fence = Q1 - 1.5 * IQR
upper_fence = Q3 + 1.5 * IQR

print(f"Q1 (25th percentile): {Q1:,.0f}")
print(f"Q3 (75th percentile): {Q3:,.0f}")
print(f"IQR (Q3 - Q1):        {IQR:,.0f}")
print(f"Lower fence:          {lower_fence:,.0f}")
print(f"Upper fence:          {upper_fence:,.0f}")

outliers = df[(df['enrollment'] < lower_fence) | (df['enrollment'] > upper_fence)]
print(f"\nIQR outliers: {len(outliers)} of {len(df)} ({100*len(outliers)/len(df):.1f}%)")

Q1 (25th percentile): 30
Q3 (75th percentile): 200
IQR (Q3 - Q1):        170
Lower fence:          -225
Upper fence:          455

IQR outliers: 67 of 493 (13.6%)


In [6]:
mean = df['enrollment'].mean()
std = df['enrollment'].std()
df['z_score'] = (df['enrollment'] - mean) / std

z_outliers = df[df['z_score'].abs() > 3]

print(f"Mean:            {mean:,.1f}")
print(f"Std deviation:   {std:,.1f}")
print(f"Median:          {df['enrollment'].median():,.1f}")
print(f"\nZ-score outliers (|z| > 3): {len(z_outliers)}")
print(f"IQR outliers (1.5×IQR):     {len(outliers)}")
print(f"\nDifference: IQR flags {len(outliers) - len(z_outliers)} more records")

Mean:            2,121.5
Std deviation:   36,135.0
Median:          70.0

Z-score outliers (|z| > 3): 1
IQR outliers (1.5×IQR):     67

Difference: IQR flags 66 more records


In [7]:
for stype in ['INTERVENTIONAL', 'OBSERVATIONAL']:
    subset = df[df['study_type'] == stype]
    q1 = subset['enrollment'].quantile(0.25)
    q3 = subset['enrollment'].quantile(0.75)
    iqr = q3 - q1
    upper = q3 + 1.5 * iqr
    flagged = subset[subset['enrollment'] > upper]

    print(f"\n{'='*55}")
    print(f"{stype}  (n={len(subset)})")
    print(f"  Median:      {subset['enrollment'].median():>10,.0f}")
    print(f"  Q1:          {q1:>10,.0f}")
    print(f"  Q3:          {q3:>10,.0f}")
    print(f"  IQR:         {iqr:>10,.0f}")
    print(f"  Upper fence: {upper:>10,.0f}")
    print(f"  Outliers:    {len(flagged):>10} ({100*len(flagged)/len(subset):.1f}%)")


INTERVENTIONAL  (n=396)
  Median:              60
  Q1:                  24
  Q3:                 154
  IQR:                130
  Upper fence:        350
  Outliers:            48 (12.1%)

OBSERVATIONAL  (n=97)
  Median:             180
  Q1:                  50
  Q3:                 566
  IQR:                516
  Upper fence:      1,340
  Outliers:            12 (12.4%)


In [8]:
# The stratified outliers, with context
outlier_list = []
for stype in ['INTERVENTIONAL', 'OBSERVATIONAL']:
    subset = df[df['study_type'] == stype]
    q3 = subset['enrollment'].quantile(0.75)
    iqr = q3 - subset['enrollment'].quantile(0.25)
    upper = q3 + 1.5 * iqr
    flagged = subset[subset['enrollment'] > upper].copy()
    flagged['fence'] = round(upper)
    outlier_list.append(flagged)

all_outliers = pd.concat(outlier_list).sort_values('enrollment', ascending=False)
all_outliers[['nct_id', 'study_type', 'status', 'phase', 'enrollment', 'fence', 'title']].head(15)

,nct_id,study_type,status,phase,enrollment,fence,title
314,NCT02014610,OBSERVATIONAL,UNKNOWN,NaN,800000,1340,White Blood Cell Counts and Onset of Cardiovas...
315,NCT03900910,INTERVENTIONAL,RECRUITING,NA,50000,350,Gastric Cancer Prevention for Indigenous Peoples
59,NCT00125008,INTERVENTIONAL,COMPLETED,PHASE4,37673,350,Evaluation of the Vi Polysaccharide Vaccine Ag...
98,NCT03359876,OBSERVATIONAL,COMPLETED,NaN,16000,1340,Clinical Outcomes Among Non-valvular Atrial Fi...
263,NCT06117618,INTERVENTIONAL,COMPLETED,NA,12284,350,Sepsis Electronic Prompting for Timely Interve...
291,NCT07448090,OBSERVATIONAL,RECRUITING,NaN,10000,1340,Real-World Case Registry Study on Obesity
395,NCT04212520,OBSERVATIONAL,RECRUITING,NaN,6500,1340,Predictive Value of Non-fasting Blood Lipid Le...
36,NCT00341835,OBSERVATIONAL,COMPLETED,NaN,6356,1340,Genetic Epidemiology of Lung Cancer
142,NCT01891240,OBSERVATIONAL,UNKNOWN,NaN,5000,1340,IMproved PRegnancy Outcome by Early Detection
63,NCT05009225,OBSERVATIONAL,UNKNOWN,NaN,4000,1340,Clinical Decision Support for Atrial Fibrillat...


## Q4 — Enrollment Performance & Outlier Detection

**Question:** Trends in patient enrollment across trial types; flag implausible or outlier
values and explain the detection approach.

### Distribution shape

Enrollment is extremely right-skewed: **mean 2,121 vs median 70** (a 30× gap), driven by a
long upper tail reaching 800,000. Half of all trials enroll 70 or fewer participants.

### Method selection (validated empirically, not asserted)

Z-score detection flagged only **1** outlier; IQR flagged **67**. The z-score failed because
its standard deviation (36,135) was itself inflated by the extreme values, letting them
escape the 3σ threshold — the outliers corrupted the yardstick meant to catch them. IQR uses
position-based quartiles, which extreme values cannot shift, making it the correct tool for
skewed data. Multiplier: Tukey's conventional 1.5× (a judgment-based constant, not derived).

### Stratified detection — populations differ ~4× in scale

| Type | n | Median | Q1 | Q3 | Upper fence | Outliers |
|------|---|--------|----|----|-------------|----------|
| INTERVENTIONAL | 396 | 60 | 24 | 154 | 350 | 48 (12.1%) |
| OBSERVATIONAL | 97 | 180 | 50 | 566 | 1,340 | 12 (12.4%) |

A 500-participant study is *typical* for observational research (below its Q3 of 566) but a
clear outlier for interventional research (past its fence of 350) — the same value carries
opposite meaning by type. Pooled detection (single fence of 455) produced 67 flags; stratified
detection produced 60, removing 7 false positives where normal-sized observational studies were
misjudged against the interventional scale. The near-identical ~12% outlier rate in both
populations confirms the method is calibrated fairly to each.

### Outlier vs. implausible — the critical distinction

Of the flagged outliers, **none are implausible**. On inspection they comprise:
- Observational registries (max 800,000 — a cardiovascular records study; a "Real-World Case
  Registry" on obesity at 10,000)
- Vaccine trials (37,673 — Vi polysaccharide vaccine)
- Population-prevention programs (50,000 — gastric cancer prevention)
- Non-pharmaceutical system interventions (sepsis electronic prompting at 12,284; clinical
  risk scores)

These reflect legitimate study-design diversity, not data errors. **Naive outlier removal
would erroneously delete valid large studies** — the reason the outlier-vs-implausible
distinction matters. No implausible values (negatives, impossible magnitudes) were found.
Zero-enrollment studies (19) correspond exactly to WITHDRAWN trials (validated in profiling).

### Answer to Q4

Enrollment scales with study type: observational studies (median 180) run larger than
interventional (median 60), and both distributions are heavily right-skewed. Statistical
outliers exist in both, but every flagged record is a legitimate large study rather than a
data error. The methodological takeaway is that outlier detection here must be stratified by
study type and interpreted against domain plausibility, not applied blindly.

In [6]:
query = """
SELECT
    l.country,
    COUNT(DISTINCT l.study_id) AS trial_count
FROM locations l
--WHERE l.country IS NOT NULL
GROUP BY l.country
ORDER BY trial_count DESC
LIMIT 30
"""
pd.read_sql(query, conn)

,country,trial_count
0,United States,162
1,China,45
2,Canada,34
3,France,32
4,Germany,27
5,Italy,27
6,Spain,27
7,United Kingdom,21
8,Turkey (Türkiye),17
9,Australia,15


In [11]:
query = """
SELECT
l.country,
COUNT(DISTINCT l.study_id) AS trial_count,
s.study_type
FROM LOCATIONS l
    JOIN STUDIES s ON l.study_id = s.study_id
GROUP BY l.country, STUDY_TYPE
ORDER BY country DESC
"""
pd.read_sql(query, conn)

,country,trial_count,study_type
0,Zambia,1,INTERVENTIONAL
1,United States,138,INTERVENTIONAL
2,United States,24,OBSERVATIONAL
3,United Kingdom,17,INTERVENTIONAL
4,United Kingdom,4,OBSERVATIONAL
...,...,...,...
100,Australia,13,INTERVENTIONAL
101,Australia,2,OBSERVATIONAL
102,Armenia,1,INTERVENTIONAL
103,Argentina,3,INTERVENTIONAL


In [18]:
query = """
Select count(distinct study_id) from locations
"""
pd.read_sql(query, conn)

,count(distinct study_id)
0,455


In [12]:
# Consolidated evidence for the Q5 geographic coverage caveat
query = """
SELECT
    (SELECT COUNT(*) FROM studies) AS total_studies,
    (SELECT COUNT(DISTINCT study_id) FROM locations) AS with_locations,
    (SELECT COUNT(*) FROM studies)
      - (SELECT COUNT(DISTINCT study_id) FROM locations) AS without_locations,
    (SELECT COUNT(*)
     FROM studies s
     LEFT JOIN (SELECT DISTINCT study_id FROM locations) l ON s.study_id = l.study_id
     WHERE s.status = 'COMPLETED'
       AND s.study_type = 'INTERVENTIONAL'
       AND l.study_id IS NULL) AS completed_interventional_no_location
"""
pd.read_sql(query, conn)

,total_studies,with_locations,without_locations,completed_interventional_no_location
0,500,455,45,14


In [8]:

query = """
SELECT
        phase,
        start_date,
        completion_date,
        -- Only compute duration where BOTH dates are full (10-char) for precision
        CASE
            WHEN LENGTH(start_date) = 10 AND LENGTH(completion_date) = 10
            THEN (julianday(completion_date) - julianday(start_date)) / 365.25
            ELSE NULL
        END AS duration_years
    FROM studies
    WHERE study_type = 'INTERVENTIONAL'
      AND phase IS NOT NULL AND phase != 'NA'
      AND start_date IS NOT NULL AND completion_date IS NOT NULL
"""
pd.read_sql(query,conn)

,phase,start_date,completion_date,duration_years
0,PHASE3,2021-02-08,2021-05-25,0.290212
1,PHASE2,2023-09-21,2028-09-21,5.002053
2,PHASE1,2009-01,2009-07,NaN
3,PHASE1,2024-07,2025-12,NaN
4,PHASE1,2017-03-22,2017-10-02,0.531143
...,...,...,...,...
185,PHASE2,2021-02-04,2025-06-30,4.399726
186,PHASE1,2019-12-03,2020-07-16,0.618754
187,PHASE2,2008-01,2008-02,NaN
188,PHASE1,2011-01,2011-08,NaN


In [7]:
query = """
WITH durations AS (
    SELECT
        phase,
        start_date,
        completion_date,
        -- Only compute duration where BOTH dates are full (10-char) for precision
        CASE
            WHEN LENGTH(start_date) = 10 AND LENGTH(completion_date) = 10
            THEN (julianday(completion_date) - julianday(start_date)) / 365.25
            ELSE NULL
        END AS duration_years
    FROM studies
    WHERE study_type = 'INTERVENTIONAL'
      AND phase IS NOT NULL AND phase != 'NA'
      AND start_date IS NOT NULL AND completion_date IS NOT NULL
)
SELECT
    phase,
    COUNT(*)                                   AS trials_with_both_dates,
    SUM(CASE WHEN duration_years IS NOT NULL THEN 1 ELSE 0 END) AS usable_for_duration,
    ROUND(AVG(duration_years), 2)              AS avg_duration_years,
    ROUND(MIN(duration_years), 2)              AS min_years,
    ROUND(MAX(duration_years), 2)              AS max_years
FROM durations
GROUP BY phase
ORDER BY phase
"""
pd.read_sql(query, conn)

,phase,trials_with_both_dates,usable_for_duration,avg_duration_years,min_years,max_years
0,EARLY_PHASE1,3,1,2.03,2.03,2.03
1,PHASE1,63,41,2.11,0.06,7.50
2,PHASE2,50,30,3.81,0.45,24.85
3,PHASE3,34,19,3.41,0.13,11.91
4,PHASE4,40,13,1.74,0.00,5.33


In [12]:
query = """
SELECT
    COUNT(*) AS total_interventional_with_phase,
    SUM(CASE WHEN start_date IS NOT NULL AND completion_date IS NOT NULL
             THEN 1 ELSE 0 END) AS has_both_dates,
    SUM(CASE WHEN LENGTH(start_date) = 10 AND LENGTH(completion_date) = 10
             THEN 1 ELSE 0 END) AS has_both_full_dates
FROM studies
WHERE study_type = 'INTERVENTIONAL' AND phase IS NOT NULL AND phase != 'NA'
"""
pd.read_sql(query, conn)

,total_interventional_with_phase,has_both_dates,has_both_full_dates
0,201,190,104


## Q5 — Geographic & Duration Insights

### Geographic distribution
Trials concentrate heavily in the United States (162, ~3.5× the next country: China 45,
Canada 34, France 32). This reflects both genuine US research infrastructure and
**registry-source bias** — ClinicalTrials.gov is a US registry, so US trials are
systematically over-represented. Findings describe trials *registered on this platform*,
not global trial activity. (Counts exceed 500 because multinational trials appear in each
host country.)

**Coverage caveat:** based on 455 of 500 studies (91%) with location data; 45 lack it,
including 14 completed interventional trials (a source-records gap, verified not a
referential-integrity failure). Geographic findings are biased toward better-maintained records.

### Duration by phase
| Phase | Usable n | Avg duration (yrs) |
|-------|----------|--------------------|
| PHASE1 | 41 | 2.11 |
| PHASE2 | 30 | 3.81 |
| PHASE3 | 19 | 3.41 |
| PHASE4 | 13 | 1.74 |
| EARLY_PHASE1 | 1 | (excluded — n=1) |

Pattern is consistent with clinical development: Phase 1 (short safety studies) < Phase 2/3
(longer efficacy studies). Phase 4 shortest, consistent with focused post-marketing monitoring.

**Critical confidence caveat — the date-format defect:** of 201 phased interventional trials,
190 have both dates but only **104 (52%)** have both at day-level precision. The other 86 carry
year-month dates where duration would carry up to ±60 days/date of error. Duration was computed
only on the 104 full-precision trials rather than fabricate a day. Consequently, per-phase
samples are small (Phase 4: n=13) and averages are indicative, not definitive. This is the
direct analytical cost of the mixed-date-format defect (DEF-002).